# Football Text Intelligence 360 — Training + Temporal Evaluation (ONE NOTEBOOK)

**Phạm vi:** Thí nghiệm, lựa chọn model, diagnostics, tracking MLflow. **Không publish PostgreSQL, không tạo mart Superset và không thực hiện inference production.**

**Luồng:** 5 nguồn Parquet → data quality → topic clustering/NMF → sentiment experiments → evaluation theo thời gian → anomaly detection → MLflow runs và artifacts. `football-lake` sẽ tự tải model đã duyệt để suy luận và đẩy DWH.

**GPU:** tự dò CUDA; mô hình Transformers (nếu bật) dùng GPU, các mô hình scikit-learn hiện tại chạy CPU; có thêm PyTorch sentiment candidate train CUDA vì không có CUDA backend trong sklearn. Từng run ghi rõ thiết bị. Không ngầm nhận rằng CPU model là GPU model.


## A. Cấu hình và cài môi trường

Template gốc dùng **Colab + `colab_utils.install_tailscale()` / `setup_tailscale()` / `setup_mlflow()`**, MinIO qua DuckDB `httpfs`, MLflow Tracking Server qua Tailscale. Notebook này giữ các điểm đó, nhưng không chép access key/secret vào notebook. Upload `colab_utils.py` theo template khi chạy Colab, hoặc đảm bảo Tailscale đã kết nối trước ở môi trường cục bộ.

Cần có: `mlflow==3.16.1`, `duckdb`, `pyarrow`, `scikit-learn`, `pandas`, `scipy`, `matplotlib`, `joblib`, `pysocks`, `transformers` + `torch` **chỉ nếu bật Transformer**.

**Không lưu credential, access token, tên tài khoản YouTube hoặc author_hash lên MLflow.** Dataset nội dung được lưu ở dạng tối giản (doc ID, text, url, thời gian, metadata không định danh tác giả).

In [ ]:
# Chạy ô này 1 lần trong Colab/Jupyter; bỏ qua nếu đã cài đủ môi trường.
import importlib.util, subprocess, sys
REQUIRED = {'mlflow':'mlflow==3.16.1','duckdb':'duckdb>=1.1','pyarrow':'pyarrow>=15',
            'sklearn':'scikit-learn>=1.3','pandas':'pandas>=2','scipy':'scipy>=1.10',
            'matplotlib':'matplotlib>=3.7','joblib':'joblib>=1.3','socks':'pysocks'}
missing = [p for module,p in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    print('Installing:', missing)
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('Dependencies ready; restart the kernel if pip updated a loaded package.')

In [ ]:
import os, sys, json, hashlib, html, re, time, warnings, tempfile, platform
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import duckdb
import scipy
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import NMF
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import silhouette_score, f1_score, accuracy_score, classification_report, confusion_matrix
from scipy.spatial.distance import jensenshannon
from joblib import dump

# === CHỈNH CẤU HÌNH TẠI ĐÂY ===
SEED = 42
np.random.seed(SEED)
EXPERIMENT_NAME = os.getenv('FOOTBALL_TEXT_EXPERIMENT', 'Football_Text_Intelligence_360')
# Địa chỉ trong template của bạn; có thể override bằng Colab Secret MLFLOW_TRACKING_URI
DEFAULT_TRACKING_URI = 'http://100.116.34.36:8080'
MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT', '20.41.113.183:9000')
S3_ROOT = os.getenv('FOOTBALL_TEXT_S3_ROOT', 's3://football-lake/dwh')
LOCAL_DATA_DIR = Path(os.getenv('FOOTBALL_TEXT_DATA_DIR', '/content' if 'google.colab' in sys.modules else '/mnt/data'))
WORKDIR = Path(os.getenv('FOOTBALL_TEXT_WORKDIR','./football_text_output')).resolve()
WORKDIR.mkdir(parents=True,exist_ok=True)
FILES = ['mart_ml_text_google_news','mart_ml_text_football_news',
         'mart_ml_text_youtube_videos','mart_ml_text_youtube_comments','mart_ml_text_wikipedia']
K_CANDIDATES = [6,10,14]
MAX_TOPIC_FIT_DOCS = 6500
TOPIC_SAMPLE_FOR_SILHOUETTE = 550
RUN_NMF_COMPARISON = True
ENABLE_TRANSFORMER_TEACHER = False  # bật nếu runtime tải được model pretrained; tốn thời gian/GPU
TRANSFORMER_MODEL = 'cardiffnlp/twitter-xlm-roberta-base-sentiment'
TRANSFORMER_MAX_RECORDS = 6000  # ưu tiên 30 ngày gần nhất; nếu bật, ít dữ liệu hơn có thể không phủ toàn bộ
ENABLE_STUDENT_SENTIMENT = True
HUMAN_LABELS_CSV = Path(os.getenv('HUMAN_LABELS_CSV','human_sentiment_labels.csv'))
# Tiêu đề + text dùng cho BI; không xuất author, author_hash hay nội dung Wikipedia đầy đủ vào snapshot scored.

In [ ]:
# Tailscale/MLflow: giữ tinh thần template. Secrets lấy từ env hoặc Colab userdata.
def get_secret(key, default=None):
    v=os.getenv(key)
    if v:return v
    try:
        from google.colab import userdata
        return userdata.get(key) or default
    except Exception:
        return default

MLFLOW_URI = get_secret('MLFLOW_TRACKING_URI', DEFAULT_TRACKING_URI)
IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    try:
        import colab_utils
    except ImportError:
        from google.colab import files
        print('Hãy upload colab_utils.py đã dùng trong template gốc:')
        files.upload()
        import importlib
        importlib.invalidate_caches()
        import colab_utils
    colab_utils.install_tailscale()
    colab_utils.setup_tailscale()
    if not colab_utils.tailscale_is_connected():
        raise RuntimeError('Tailscale chưa kết nối; dừng trước khi training để tránh log thất lạc.')
    colab_utils.setup_mlflow(server_uri=MLFLOW_URI,experiment=EXPERIMENT_NAME)
mlflow.set_tracking_uri(MLFLOW_URI)
try:
    _client=MlflowClient()
    _client.search_experiments(max_results=1)  # fail closed
    mlflow.set_experiment(EXPERIMENT_NAME)
except Exception as e:
    raise RuntimeError(f'Không kết nối được MLflow server {MLFLOW_URI}; kiểm tra Tailscale. Không fallback về local.') from e
print('MLflow connected:',MLFLOW_URI,'experiment=',EXPERIMENT_NAME)

## A1. Kiểm tra GPU / chiến lược thiết bị

Dò CUDA tự động. GPU phù hợp với **Transformer**; TF-IDF, MiniBatchKMeans, NMF, LogisticRegression và IsolationForest phiên bản `scikit-learn` vẫn dùng CPU. GPU không có nghĩa tất cả thuật toán tự chuyển sang CUDA. Nếu muốn thí nghiệm train Transformer, cần dữ liệu nhãn tin cậy trước khi fine-tune; notebook không tự huấn luyện trên pseudo-label rồi công bố là ground truth.


In [ ]:
import importlib.util
GPU_AVAILABLE=False
GPU_NAME=None
if importlib.util.find_spec('torch') is not None:
    import torch
    GPU_AVAILABLE=bool(torch.cuda.is_available())
    GPU_NAME=torch.cuda.get_device_name(0) if GPU_AVAILABLE else None
DEVICE='cuda:0' if GPU_AVAILABLE else 'cpu'
print('Device preference:',DEVICE,'GPU:',GPU_NAME or 'not available')
# Teacher inference mặc định bật khi CUDA có sẵn; có thể override bằng biến môi trường.
ENABLE_TRANSFORMER_TEACHER=(os.getenv('FOOTBALL_TEXT_TRANSFORMER_TEACHER',
                                     '1' if GPU_AVAILABLE else '0').strip().lower() in ('1','true','yes'))
print('Transformer teacher enabled:',ENABLE_TRANSFORMER_TEACHER)


## B. Đọc 5 Parquet + data quality

Ưu tiên đọc file người dùng tải lên trong `LOCAL_DATA_DIR`. Nếu thiếu, đọc từ MinIO với `duckdb httpfs` bằng `MINIO_ACCESS_KEY` và `MINIO_SECRET_KEY` ở Colab Secrets (tương tự template). `published_at` là **thời điểm nội dung được đăng**; `first_seen_date`/`ingest_date` chỉ phục vụ quan sát ingest/backfill.

In [ ]:
def minio_con():
    con=duckdb.connect(':memory:')
    con.execute('INSTALL httpfs')
    con.execute('LOAD httpfs')
    ak=get_secret('MINIO_ACCESS_KEY');sk=get_secret('MINIO_SECRET_KEY')
    if not ak or not sk:
        raise RuntimeError('Thiếu MINIO_ACCESS_KEY/MINIO_SECRET_KEY trong Secrets')
    # COPY cách escape từ template: không print chuỗi SQL có credentials.
    q=lambda x: str(x).replace("'","''")
    con.execute(f"""CREATE OR REPLACE SECRET minio_secret (TYPE S3,
        KEY_ID '{q(ak)}', SECRET '{q(sk)}', ENDPOINT '{q(MINIO_ENDPOINT)}',
        URL_STYLE 'path', USE_SSL false)""")
    return con

def load_sources():
    raws={};source_paths={};con=None
    for name in FILES:
        p=LOCAL_DATA_DIR/(name+'.parquet')
        if p.exists():
            raws[name]=pd.read_parquet(p)
            source_paths[name]=str(p.resolve())
        else:
            if con is None:con=minio_con()
            s3=f'{S3_ROOT}/{name}.parquet'
            raws[name]=con.execute('SELECT * FROM read_parquet(?)',[s3]).df()
            source_paths[name]=s3
        print(f'{name}: {len(raws[name]):,} records, {len(raws[name].columns)} columns')
    if con is not None:con.close()
    return raws,source_paths

raws,source_paths=load_sources()
assert all(len(raws[k])>0 for k in FILES)

In [ ]:
def to_datetime_utc(s):
    if pd.api.types.is_integer_dtype(s):
        nonnull=s.dropna()
        if len(nonnull) and nonnull.abs().median()>10**13:
            return pd.to_datetime(s,unit='us',utc=True,errors='coerce')
    return pd.to_datetime(s,utc=True,errors='coerce')

def normalize_text(s):
    if pd.isna(s):return ''
    txt=html.unescape(str(s))
    txt=re.sub(r'https?://\S+', ' ',txt)
    txt=re.sub(r'\s+', ' ',txt)
    return txt.strip()

def detect_lang(text,hint=None):
    if isinstance(hint,str) and hint.lower() in ('vi','en'):return hint.lower()
    low=text.lower()
    if re.search(r'[ăâđêôơưĂÂĐÊÔƠƯ]',text) or len(re.findall(r'\b(?:đội|trận|bóng|đá|cầu|thủ|huấn|luyện|chiến|thắng|tuyển|vô|địch|chuyển|nhượng)\b',low))>=2:
        return 'vi'
    return 'und' # không tự gán English cho tiếng Indonesia/Spanish/Arabic

VN_TERMS=['việt nam','vietnam','v-league','vleague','v.league','hà nội fc','hanoi fc',
          'đội tuyển việt nam','bóng đá việt nam','u23 việt nam','hoàng anh gia lai','thể công viettel','nam định fc','hải phòng fc']
INTL_TERMS=['premier league','champions league','la liga','bundesliga','serie a','ligue 1','arsenal','chelsea','liverpool','barcelona','real madrid','manchester united','man city','bayern','psg','inter milan','juventus','tottenham','cristiano ronaldo','messi']
def scope_guess(s):
    txt=s.lower()
    if any(k in txt for k in VN_TERMS):return 'domestic_vn'
    if any(k in txt for k in INTL_TERMS):return 'international'
    return 'unknown'

def col(d,name,default=''):
    return d[name] if name in d else pd.Series([default]*len(d),index=d.index)

def normalize_one(key,d):
    # Giữ nguyên khóa và provenance; không làm giả ngôn ngữ/quốc gia.
    if key=='mart_ml_text_google_news':
        src='google_news'; ident=col(d,'news_id');title=col(d,'title');body=col(d,'summary');lang=col(d,'language');
        url=col(d,'url');publisher=col(d,'publisher');text=col(d,'text')
        event=col(d,'published_at');video=col(d,'video_id');engagement=pd.Series([0]*len(d))
    elif key=='mart_ml_text_football_news':
        src='football_news';ident=col(d,'news_id');title=col(d,'title');body=col(d,'text');lang=col(d,'language');
        url=col(d,'url');publisher=col(d,'source_name');text=col(d,'text')
        event=col(d,'published_at');video=col(d,'video_id');engagement=pd.Series([0]*len(d))
    elif key=='mart_ml_text_youtube_videos':
        src='youtube_video';ident=col(d,'video_id');title=col(d,'title');body=col(d,'description');lang=col(d,'language');
        url=pd.Series(['https://www.youtube.com/watch?v='+str(v) for v in ident],index=d.index)
        publisher=col(d,'channel_title');text=col(d,'text');event=col(d,'published_at');
        video=col(d,'video_id');engagement=col(d,'n_comments_collected',0)
    elif key=='mart_ml_text_youtube_comments':
        src='youtube_comment';ident=col(d,'comment_id');title=col(d,'video_title');body=col(d,'text');lang=col(d,'language');
        video=col(d,'video_id');url=pd.Series(['https://www.youtube.com/watch?v='+str(v) for v in video],index=d.index)
        publisher=col(d,'video_channel_title');text=col(d,'text');event=col(d,'published_at');engagement=col(d,'like_count',0)
    else: raise ValueError(key)
    data=pd.DataFrame({
        'doc_id':src+':'+ident.astype(str), 'source_type':src,
        'title':title.fillna('').astype(str), 'body':body.fillna('').astype(str),
        'raw_text':text.fillna('').astype(str), 'publisher':publisher.fillna('').astype(str),
        'url':url.fillna('').astype(str), 'video_id':video.fillna('').astype(str),
        'lang_hint':lang.fillna('').astype(str),'published_at':to_datetime_utc(event).values,
        'engagement':pd.to_numeric(engagement,errors='coerce').fillna(0).clip(lower=0).values
    })
    data['text_clean']=data['raw_text'].map(normalize_text)
    data['title_clean']=data['title'].map(normalize_text)
    data['body_clean']=data['body'].map(normalize_text)
    if src=='youtube_comment':
        data['topic_input']=(data['title_clean']+' '+data['text_clean'].str.slice(0,180)).str.slice(0,800)
        data['sentiment_input']=data['text_clean'].str.slice(0,500)
    else:
        data['topic_input']=(data['title_clean']+' '+data['body_clean'].str.slice(0,700)).str.slice(0,1000)
        data['sentiment_input']=(data['title_clean']+' '+data['body_clean'].str.slice(0,200)).str.slice(0,500)
    data['language']= [detect_lang(t,h) for t,h in zip(data['text_clean'],data['lang_hint'])]
    data['scope']=[scope_guess(t) for t in data['topic_input']]
    return data

raw_count={k:len(v) for k,v in raws.items()}
frames=[normalize_one(k,raws[k]) for k in FILES if k!='mart_ml_text_wikipedia']
docs=pd.concat(frames,ignore_index=True)
rows_before=len(docs)
docs=docs.loc[docs['published_at'].notna() & docs['text_clean'].str.len().gt(0)].copy()
docs=docs.drop_duplicates('doc_id',keep='last').reset_index(drop=True)
assert not docs.doc_id.duplicated().any()
docs['published_at']=pd.to_datetime(docs['published_at'],utc=True)
docs['event_date']=docs['published_at'].dt.floor('D').dt.tz_localize(None)
ref_date=docs['event_date'].max()
docs['days_old']=(ref_date-docs['event_date']).dt.days
print('Scored document candidates:',len(docs),'of',rows_before,'reference date:',ref_date.date())
print(docs.groupby('source_type').agg(rows=('doc_id','size'),min_date=('event_date','min'),max_date=('event_date','max')).to_string())
print('30-day source coverage:')
print(docs.loc[docs.days_old.between(0,29)].source_type.value_counts().to_string())
print('Language:',docs.language.value_counts().to_dict(), 'scope:',docs.scope.value_counts().to_dict())

In [ ]:
# Input snapshot / DQ + dataset manifest: chạy trên nguồn thực, không đoán số từ template.
quality={
 'raw_rows':raw_count,'scored_docs':len(docs),'dropped_invalid_or_duplicate':rows_before-len(docs),
 'reference_date':str(ref_date.date()),'language_distribution':docs.language.value_counts().to_dict(),
 'scope_distribution':docs.scope.value_counts().to_dict(),
 'source_distribution':docs.source_type.value_counts().to_dict(),
 'note':'Language/scope labels outside Google News are heuristic. Wikipedia is reference only.'
}
for name,df in raws.items():
    quality[name+'_columns']=list(df.columns)
    quality[name+'_null_rates']={c:float(x) for c,x in df.isna().mean().items()}
    quality[name+'_rows']=len(df)

# Không export author, author_hash hay text Wikipedia vào bộ scored.
REF_META=raws['mart_ml_text_wikipedia'][['title','category','language']].copy()
print(pd.DataFrame([{'source':k,'rows':v} for k,v in raw_count.items()]).to_string(index=False))

## C. Topic clustering — thí nghiệm nhiều K + NMF đối chứng

Chỉ fit chủ đề trên **news/video** (bình luận ngắn có ngữ cảnh của tiêu đề video được gán topic lúc inference), tránh bình luận một từ kéo lệch centroid. Fit TF-IDF **chỉ trên train thời gian**, lựa chọn K qua silhouette cosine và kích thước cụm; validation/test dùng đo drift (không phải supervised accuracy). NMF là hướng đối chứng để phân tích chủ đề nhưng **KMeans** được dùng làm mã topic ổn định cho dashboard.

In [ ]:
# Đảm bảo 1 ngày không bị chia qua nhiều tập.
trainable=docs[docs.source_type.isin(['google_news','football_news','youtube_video'])].copy()
trainable=trainable.loc[trainable.topic_input.str.len().ge(20)].sort_values('event_date').reset_index(drop=True)
unique_dates=np.array(sorted(trainable.event_date.unique()))
if len(unique_dates)<5:raise RuntimeError('Quá ít ngày để chia thời gian; kiểm tra published_at')
# Cutoffs theo quantile SỐ DÒNG, nhưng không tách các bản ghi có cùng ngày.
# Đừng chia theo số ngày duy nhất: lịch sử dài có thể khiến train chỉ có vài trăm dòng.
train_cut=trainable.event_date.quantile(.70).normalize()
valid_cut=trainable.event_date.quantile(.85).normalize()
if train_cut>=valid_cut:raise RuntimeError('Train/valid cutoffs overlap: check daily data concentration')
trainable['split']=np.select([trainable.event_date<=train_cut,trainable.event_date<=valid_cut],['train','valid'],default='test')
print('date boundaries:',train_cut,valid_cut)
print(trainable.split.value_counts())
assert trainable.split.nunique()==3
train_text=trainable.loc[trainable.split.eq('train'),'topic_input'].astype(str)
if len(train_text)>MAX_TOPIC_FIT_DOCS:
    train_text=train_text.sample(MAX_TOPIC_FIT_DOCS,random_state=SEED)
vectorizer=TfidfVectorizer(max_features=20000,min_df=3,max_df=.95,ngram_range=(1,2),
                           token_pattern=r'(?u)\b\w\w+\b',sublinear_tf=True,dtype=np.float32)
Xtopic=vectorizer.fit_transform(train_text)
if Xtopic.shape[1]<20:raise RuntimeError('Too few TF-IDF terms: check input and min_df')
print('Topic TF-IDF:',Xtopic.shape)

In [ ]:
# Create parent MLflow run; tất cả thử nghiệm/logging nằm dưới run này.
parent=mlflow.start_run(run_name=f'text360_{ref_date:%Y%m%d}_seed{SEED}')
PARENT_RUN_ID=parent.info.run_id
mlflow.set_tags({'project':'football-lake','pipeline':'football-text-360','split':'chronological_dates',
                 'label_status':'unlabeled/weak','reference_date':str(ref_date.date()),
                 'privacy':'author/author_hash excluded','upstream':'5 text parquet sources'})
mlflow.log_params({'device_preference':DEVICE,'gpu_detected':str(GPU_AVAILABLE),'sklearn_training_device':'cpu','seed':SEED,'candidate_k':','.join(map(str,K_CANDIDATES)),
                   'topic_train_end':str(pd.Timestamp(train_cut).date()),
                   'topic_valid_end':str(pd.Timestamp(valid_cut).date()),
                   'max_topic_fit_docs':MAX_TOPIC_FIT_DOCS,
                   'nmf_comparison':RUN_NMF_COMPARISON,
                   'teacher_enabled':ENABLE_TRANSFORMER_TEACHER,
                   'sentiment_label_mode':'weak_lexicon_unless_teacher_enabled'})
mlflow.log_dict(quality,'quality/quality_report.json')
mlflow.log_dict({'source_paths':source_paths,'files':FILES,
                 'raw_rows':raw_count,'schema_by_source':{k:list(v.columns) for k,v in raws.items()}},
                'data/source_manifest.json')
# Dataset đầy đủ ở Data Lake; MLflow chỉ log lineage, fingerprint và schema (không duplicate data).
for name in FILES:
    p=LOCAL_DATA_DIR/(name+'.parquet')
    if p.exists():
        # SHA-256 nguồn: nguyên bản dữ liệu không log để tránh tác giả/bình luận viên bị lộ định danh.
        digest=hashlib.file_digest(open(p,'rb'),'sha256').hexdigest()
        mlflow.set_tag('source_sha256_'+name[:60],digest)
for split_name, part in trainable.groupby('split'):
    mlflow.log_input(mlflow.data.from_pandas(part[['doc_id','event_date','source_type']],
                source='football-text-normalized',name=f'topics_{split_name}'),
                context=split_name)
print('Parent MLflow run:',PARENT_RUN_ID)

In [ ]:
# Có thể hơi lâu ở Google Colab CPU. Kết quả mọi candidate là nested MLflow run.
K_CANDIDATES=[k for k in K_CANDIDATES if k<len(train_text)//10]
if not K_CANDIDATES:raise ValueError('Not enough topic training docs for clustering')
topic_candidates=[];fitted_clusters={}
for k in K_CANDIDATES:
    with mlflow.start_run(run_name=f'Topic_MiniBatchKMeans_k{k}',nested=True):
        st=time.time()
        km=MiniBatchKMeans(n_clusters=k,random_state=SEED,batch_size=512,n_init=4,max_iter=120)
        labels=km.fit_predict(Xtopic)
        ids=np.random.default_rng(SEED).choice(len(labels),size=min(TOPIC_SAMPLE_FOR_SILHOUETTE,len(labels)),replace=False)
        sil=float(silhouette_score(Xtopic[ids],labels[ids],metric='cosine')) if len(set(labels[ids]))>1 else -1.0
        min_frac=float(np.bincount(labels,minlength=k).min()/len(labels))
        elapsed=time.time()-st
        mlflow.log_params({'algorithm':'MiniBatchKMeans','k':k,'vectorizer':'word_tf_idf_1_2',
                           'min_df':3,'max_features':20000,'n_train':len(train_text),
                           'sample_silhouette':len(ids)})
        mlflow.log_metrics({'silhouette_cosine':sil,'inertia':float(km.inertia_),
                            'smallest_cluster_share':min_frac,'train_seconds':elapsed})
        model=Pipeline([('tfidf',vectorizer),('cluster',km)])
        mlflow.sklearn.log_model(model,name='topic_model',input_example=train_text.head(2).tolist())
        fitted_clusters[k]=km
        topic_candidates.append({'k':k,'silhouette_cosine':sil,'min_cluster_share':min_frac,'seconds':elapsed})
        print(k,round(sil,4),round(min_frac,4),'seconds',round(elapsed,1))
comparison=pd.DataFrame(topic_candidates).sort_values('silhouette_cosine',ascending=False)
# Penalize pathological tiny clusters; if all below threshold, choose best silhouette and flag low quality.
eligible=comparison.loc[comparison.min_cluster_share>=0.005]
best=(eligible if len(eligible) else comparison).iloc[0]
BEST_K=int(best.k); topic_km=fitted_clusters[BEST_K]
TOPIC_PIPELINE=Pipeline([('tfidf',vectorizer),('cluster',topic_km)])
mlflow.log_params({'chosen_topic_k':BEST_K,'topic_model_selection':'max_cosine_silhouette_with_min_cluster_share_0.005'})
mlflow.log_metrics({'selected_topic_silhouette_cosine':float(best.silhouette_cosine),
                    'selected_min_cluster_share':float(best.min_cluster_share)})
mlflow.sklearn.log_model(TOPIC_PIPELINE,name='selected_topic_model',input_example=train_text.head(2).tolist())
comparison.to_csv(WORKDIR/'topic_candidates.csv',index=False)
mlflow.log_artifact(str(WORKDIR/'topic_candidates.csv'),'evaluation')

In [ ]:
# Interpretability: top TF-IDF words in centroids; no invented human topic names.
terms=np.asarray(vectorizer.get_feature_names_out())
keywords=[]
for idx,centroid in enumerate(topic_km.cluster_centers_):
    top=terms[np.argsort(centroid)[::-1][:12]].tolist()
    keywords.append({'topic_id':int(idx),'topic_name':'Topic '+str(idx)+': '+', '.join(top[:3]),
                     'top_terms':', '.join(top),'k':BEST_K})
topic_lookup=pd.DataFrame(keywords)
docs['topic_id']=TOPIC_PIPELINE.predict(docs['topic_input'].fillna('').tolist()).astype(int)
docs=docs.merge(topic_lookup[['topic_id','topic_name']],on='topic_id',how='left',validate='many_to_one')
print(topic_lookup.to_string(index=False,max_colwidth=65))
# Drift topic across chronological train vs test (news/video)
topic_test=TOPIC_PIPELINE.predict(trainable.loc[trainable.split.eq('test'),'topic_input'])
topic_train=TOPIC_PIPELINE.predict(trainable.loc[trainable.split.eq('train'),'topic_input'])
p=np.bincount(topic_train,minlength=BEST_K).astype(float)+1e-6;q=np.bincount(topic_test,minlength=BEST_K).astype(float)+1e-6
p/=p.sum();q/=q.sum()
mlflow.log_metric('topic_train_test_js_distance',float(jensenshannon(p,q)))
mlflow.log_input(mlflow.data.from_pandas(docs[['doc_id','topic_id']],name='topic_assignments',source='predicted'),context='inference')

In [ ]:
# NMF là thí nghiệm đối chứng, không thay cluster_id đã chọn.
if RUN_NMF_COMPARISON:
    with mlflow.start_run(run_name='Topic_NMF_k8_baseline',nested=True):
        nmf=NMF(n_components=min(8,max(2,len(train_text)//10)),init='nndsvda',random_state=SEED,
                max_iter=150,alpha_W=0.01,alpha_H='same',l1_ratio=0.0)
        st=time.time();W=nmf.fit_transform(Xtopic)
        mlflow.log_params({'algorithm':'NMF','k':nmf.n_components,'init':'nndsvda','max_iter':150})
        mlflow.log_metrics({'reconstruction_error':float(nmf.reconstruction_err_),
                            'train_seconds':time.time()-st,'zero_document_topic_frac':float((W.sum(axis=1)==0).mean())})
        nmf_model=Pipeline([('tfidf',vectorizer),('nmf',nmf)])
        mlflow.sklearn.log_model(nmf_model,name='nmf_model',input_example=train_text.head(2).tolist())
        nmf_terms=[{'topic':i,'keywords':', '.join(terms[np.argsort(x)[::-1][:12]])}
                   for i,x in enumerate(nmf.components_)]
        mlflow.log_dict({'topics':nmf_terms},'evaluation/nmf_top_terms.json')

## D. Sentiment — 3 lớp, weak labeling và training student

**Hạn chế khoa học:** Không có nhãn tích cực/trung lập/tiêu cực chuẩn từ người gán nhãn trong 5 nguồn. Baseline lexicon chỉ cung cấp nhãn yếu (weak labels). `student` học các nhãn yếu đó thì F1 trên test **không đánh giá độ đúng sentiment**; chỉ log `pseudo_label_agreement_f1`. Khi có file `human_sentiment_labels.csv` (`doc_id`, `sentiment_label`) sẽ đánh giá độc lập và log `human_macro_f1`. Có thể bật pretrained transformer để lấy teacher tốt hơn, nhưng phải kiểm tra khả năng xử lý tiếng Việt và nhãn theo domain bóng đá.

In [ ]:
# Weak bilingual domain sentiment lexicon — baseline heuristic, có thể có sai lệch theo ngôn ngữ/ngữ cảnh.
POS_EN=['great','amazing','excellent','brilliant','fantastic','wonderful','love','win','victory','legend',
        'outstanding','incredible','impressive','congratulations','perfect','happy','good performance']
NEG_EN=['awful','terrible','poor','disappointing','bad','horrible','hate','injury','injured','lose','loss',
        'disaster','failure','worst','sack','embarrassing']
POS_VI=['xuất sắc','tuyệt vời','chiến thắng','hay quá','đỉnh cao','quá hay','ấn tượng','chúc mừng',
        'đẹp mắt','vui mừng','đá hay','vô địch','tốt quá','thăng hoa']
NEG_VI=['thất vọng','quá tệ','tồi tệ','chấn thương','thua đau','đá dở','kém cỏi','khủng hoảng',
        'sa thải','bất công','phẫn nộ','tức giận','đáng tiếc','thảm họa']

def weak_label(s):
    t=' '+normalize_text(s).lower()+' '
    pos=sum(t.count(k) for k in POS_EN+POS_VI)
    neg=sum(t.count(k) for k in NEG_EN+NEG_VI)
    if pos==neg:return 'neutral',0.33
    if pos>neg:return 'positive',float(min(0.8,0.40+0.1*(pos-neg)))
    return 'negative',float(min(0.8,0.40+0.1*(neg-pos)))

res=[weak_label(s) for s in docs.sentiment_input]
docs['weak_sentiment']=[x[0] for x in res]
docs['weak_confidence']=[x[1] for x in res]
docs['sentiment_label']=docs['weak_sentiment']
docs['sentiment_confidence']=docs['weak_confidence']
docs['sentiment_method']='weak_lexicon'
with mlflow.start_run(run_name='Sentiment_Weak_Lexicon_Baseline',nested=True):
    mlflow.log_params({'model_type':'bilingual_rule_lexicon','n_positive_terms':len(POS_EN+POS_VI),
                       'n_negative_terms':len(NEG_EN+NEG_VI),'human_labels':False})
    mlflow.log_metrics({'documents_scored':len(docs),
                        'positive_rate':float(docs.weak_sentiment.eq('positive').mean()),
                        'negative_rate':float(docs.weak_sentiment.eq('negative').mean()),
                        'neutral_rate':float(docs.weak_sentiment.eq('neutral').mean())})
    mlflow.log_dict({'positive_terms':POS_EN+POS_VI,'negative_terms':NEG_EN+NEG_VI,
                     'limitation':'Heuristic baseline; no ground truth, no accuracy claim'},
                     'model/weak_lexicon.json')
print('weak label distribution:',docs.weak_sentiment.value_counts(normalize=True).round(3).to_dict())

In [ ]:
# Pretrained multilingual teacher TÙY CHỌN. Khi bật, chỉ chạy trên TOP N bản ghi gần nhất;
# các bản ghi còn lại vẫn có nhãn weak_lexicon và provenance rõ ràng.
if ENABLE_TRANSFORMER_TEACHER:
    import importlib.util,subprocess
    if importlib.util.find_spec('transformers') is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers>=4.40,<5'])
    from transformers import pipeline
    import torch
    with mlflow.start_run(run_name='Sentiment_Transformer_Teacher_Inference',nested=True):
        teacher=pipeline('text-classification',model=TRANSFORMER_MODEL,
                         device=0 if torch.cuda.is_available() else -1, truncation=True)
        selected=docs.sort_values('published_at',ascending=False).head(TRANSFORMER_MAX_RECORDS)
        inputs=selected.sentiment_input.fillna('').astype(str).tolist()
        preds=teacher(inputs,batch_size=16,max_length=160,truncation=True)
        labelmap={'LABEL_0':'negative','LABEL_1':'neutral','LABEL_2':'positive',
                  'negative':'negative','neutral':'neutral','positive':'positive'}
        mapped=[labelmap.get(x['label'].lower(),labelmap.get(x['label'],'neutral')) for x in preds]
        docs.loc[selected.index,'sentiment_label']=mapped
        docs.loc[selected.index,'sentiment_confidence']=[float(x['score']) for x in preds]
        docs.loc[selected.index,'sentiment_method']='pretrained_teacher'
        mlflow.log_params({'model':TRANSFORMER_MODEL,'max_len':160,'inference_docs':len(selected)})
        mlflow.log_metric('teacher_coverage',len(selected)/len(docs))
        # Record model name/revision; log weights if storage/permission allows.
        try:
            mlflow.transformers.log_model(teacher,name='sentiment_teacher_model')
        except Exception as exc:
            mlflow.set_tag('teacher_weights_log_failure',str(exc)[:400])
            raise RuntimeError('Không log được teacher weights vào MLflow; dừng để đảm bảo tracking đầy đủ.') from exc
        print('Transformer teacher scored:',len(selected))

In [ ]:
# Student TF-IDF + Logistic Regression, train theo thời gian, đánh giá pseudo-label agreement.
# Dùng chính nguồn sentiment_method có nhãn; metric pseudo không phải Ground Truth F1.
if ENABLE_STUDENT_SENTIMENT:
    labeled=docs[docs.sentiment_input.str.len().ge(5)].sort_values('event_date').copy()
    unique_dates=np.array(sorted(labeled.event_date.unique()))
    if len(unique_dates)<5:raise ValueError('Không đủ ngày để tạo temporal train/valid/test')
    cut1=labeled.event_date.quantile(.70).normalize();cut2=labeled.event_date.quantile(.85).normalize()
    if cut1>=cut2:raise RuntimeError('Sentiment split cutoffs overlap: inspect daily volume')
    labeled['split']=np.select([labeled.event_date<=cut1,labeled.event_date<=cut2],['train','valid'],default='test')
    tr=labeled[labeled.split=='train'];va=labeled[labeled.split=='valid'];te=labeled[labeled.split=='test']
    if min(len(tr),len(va),len(te))==0:raise ValueError('Temporal split rỗng')
    if tr.sentiment_label.nunique()<2:
        print('Only one weak class in training; skipping student (no meaningful classification).')
    else:
        with mlflow.start_run(run_name='Sentiment_Student_WeakSupervision_LogReg',nested=True):
            sentiment_model=Pipeline([
                ('tfidf',TfidfVectorizer(analyzer='char',ngram_range=(3,5),min_df=3,
                                         max_features=35000,sublinear_tf=True,dtype=np.float32)),
                ('clf',LogisticRegression(max_iter=300,class_weight='balanced',random_state=SEED))])
            st=time.time();sentiment_model.fit(tr.sentiment_input,tr.sentiment_label)
            mlflow.log_params({'model':'char_tfidf_logreg','ngram_range':'3-5','class_weight':'balanced',
                'train_end':str(pd.Timestamp(cut1).date()),'valid_end':str(pd.Timestamp(cut2).date()),
                'labels':'pseudo_weak_or_teacher','train_size':len(tr),'valid_size':len(va),'test_size':len(te)})
            mlflow.log_metric('train_seconds',time.time()-st)
            for phase,part in [('valid',va),('test',te)]:
                yp=sentiment_model.predict(part.sentiment_input)
                mlflow.log_metric(phase+'_pseudo_label_agreement_macro_f1',
                         float(f1_score(part.sentiment_label,yp,labels=['negative','neutral','positive'],average='macro',zero_division=0)))
                mlflow.log_metric(phase+'_pseudo_label_agreement_accuracy',float(accuracy_score(part.sentiment_label,yp)))
            mlflow.sklearn.log_model(sentiment_model,name='sentiment_student_model',
                         input_example=tr.sentiment_input.head(2).tolist())
            for phase,part in [('train',tr),('valid',va),('test',te)]:
                snap=part[['doc_id','sentiment_input','sentiment_label','sentiment_method']]
                fp=WORKDIR/f'sentiment_{phase}.parquet';snap.to_parquet(fp,index=False)
                mlflow.log_artifact(str(fp),'datasets/sentiment_splits')
                mlflow.log_input(mlflow.data.from_pandas(snap[['doc_id','sentiment_label']],name='sentiment_'+phase),context=phase)
            # Dùng student để gán nhãn hoàn chỉnh nhằm thống nhất dashboard; giữ nhãn gốc và nguồn.
            docs['student_sentiment']=sentiment_model.predict(docs.sentiment_input)
            docs['student_confidence']=sentiment_model.predict_proba(docs.sentiment_input).max(axis=1)
            docs['sentiment_label']=docs['student_sentiment']
            docs['sentiment_confidence']=docs['student_confidence']
            docs['sentiment_method']='student_weak_supervision'
            mlflow.set_tag('sentiment_evaluation_warning','Pseudo label agreement is NOT human-ground-truth accuracy')
            print('Student weak-label trained. Need human labels for accuracy.')
            # Human validation optional; đánh giá trên labels độc lập mà không dùng train.
            if HUMAN_LABELS_CSV.exists():
                human=pd.read_csv(HUMAN_LABELS_CSV)
                assert {'doc_id','sentiment_label'}.issubset(human.columns)
                human=human.drop_duplicates('doc_id')
                human=human.loc[human.sentiment_label.isin(['negative','neutral','positive'])]
                held=docs[['doc_id','sentiment_input']].merge(human,on='doc_id',how='inner')
                if len(held)>=30 and held.sentiment_label.nunique()>=2:
                    prediction=sentiment_model.predict(held.sentiment_input)
                    mlflow.log_metric('human_label_macro_f1',float(f1_score(held.sentiment_label,prediction,
                                      labels=['negative','neutral','positive'],average='macro',zero_division=0)))
                    mlflow.log_metric('human_labeled_count',len(held))
                    mlflow.log_input(mlflow.data.from_pandas(held[['doc_id','sentiment_label']],name='human_eval'),context='evaluation')
                else: print('Human labels too few/one class; not logging human metrics.')

## D1. GPU-trained candidate — PyTorch Sentiment MLP (nếu CUDA có sẵn)

Khi có CUDA và đã train được TF-IDF student ở trên, train **thêm một neural sentiment candidate trên GPU**, dựa trên **cùng temporal split** để so sánh. Đây là weak-label learning, chưa phải supervised accuracy. GPU model và TF-IDF vectorizer được lưu thành artifacts trong cùng nested MLflow run. GPU không sẵn có → giữ sklearn CPU baseline. Model này là **candidate**; chưa tự đổi output dashboard hoặc gán alias champion.


In [ ]:
# GPU training thực sự (không chỉ GPU inference). CPU fallback đã có ở LogisticRegression.
if GPU_AVAILABLE and 'sentiment_model' in globals() and 'tr' in globals() and tr.sentiment_label.nunique()>=2:
    import torch
    import torch.nn as nn
    from sklearn.preprocessing import LabelEncoder
    import joblib
    torch.manual_seed(SEED)
    Xvec=sentiment_model.named_steps['tfidf']
    enc=LabelEncoder().fit(tr.sentiment_label)
    Xtr=Xvec.transform(tr.sentiment_input.fillna('').astype(str))
    Xva=Xvec.transform(va.sentiment_input.fillna('').astype(str))
    Xte=Xvec.transform(te.sentiment_input.fillna('').astype(str))
    Ytr=enc.transform(tr.sentiment_label)
    class SentimentMLP(nn.Module):
        def __init__(self, input_dim, n_classes):
            super().__init__()
            self.layers=nn.Sequential(nn.Linear(input_dim,64),nn.ReLU(),nn.Dropout(0.2),nn.Linear(64,n_classes))
        def forward(self,x):return self.layers(x)
    model_gpu=SentimentMLP(Xtr.shape[1],len(enc.classes_)).to('cuda')
    optim=torch.optim.AdamW(model_gpu.parameters(),lr=0.001,weight_decay=0.01)
    criterion=nn.CrossEntropyLoss()
    GPU_EPOCHS=int(os.getenv('FOOTBALL_TEXT_GPU_EPOCHS','4'))
    GPU_BATCH=int(os.getenv('FOOTBALL_TEXT_GPU_BATCH','256'))
    train_rng=np.random.default_rng(SEED)
    def eval_gpu(mat,labels):
        model_gpu.eval();pred=[]
        with torch.no_grad():
            for lo in range(0,mat.shape[0],GPU_BATCH):
                xb=torch.from_numpy(mat[lo:lo+GPU_BATCH].toarray()).float().to('cuda')
                pred.extend(model_gpu(xb).argmax(axis=1).cpu().numpy())
        return float(f1_score(labels,enc.inverse_transform(np.asarray(pred,dtype=int)),
                              average='macro',zero_division=0))
    with mlflow.start_run(run_name='Sentiment_Torch_MLP_GPU_Candidate',nested=False) as gpu_run:
        mlflow.set_tags({'project':'football-lake','candidate_type':'sentiment_weak_supervision',
                         'linked_training_run_id':PARENT_RUN_ID,'gpu_name':GPU_NAME,
                         'label_warning':'pseudo label agreement not ground truth'})
        mlflow.log_params({'algorithm':'Torch-MLP','device':'cuda:0','input_features':int(Xtr.shape[1]),
                           'epochs':GPU_EPOCHS,'batch_size':GPU_BATCH,'learning_rate':0.001,
                           'train_examples':len(tr),'classes':','.join(enc.classes_)})
        for epoch in range(GPU_EPOCHS):
            model_gpu.train();perm=train_rng.permutation(len(Ytr));loss_sum=0.
            for j in range(0,len(perm),GPU_BATCH):
                inds=perm[j:j+GPU_BATCH]
                xb=torch.from_numpy(Xtr[inds].toarray()).float().to('cuda')
                yb=torch.as_tensor(Ytr[inds],dtype=torch.long,device='cuda')
                optim.zero_grad();loss=criterion(model_gpu(xb),yb);loss.backward();optim.step()
                loss_sum+=float(loss.item())*len(inds)
            mlflow.log_metric('train_cross_entropy',loss_sum/len(Ytr),step=epoch)
            mlflow.log_metric('valid_pseudo_macro_f1',eval_gpu(Xva,va.sentiment_label),step=epoch)
        mlflow.log_metric('test_pseudo_macro_f1',eval_gpu(Xte,te.sentiment_label))
        mlflow.pytorch.log_model(model_gpu.cpu(),name='torch_sentiment_candidate')
        vec_path=WORKDIR/'gpu_sentiment_tfidf.joblib';joblib.dump(Xvec,vec_path)
        mlflow.log_artifact(str(vec_path),'preprocessor')
        mlflow.log_dict({'classes':enc.classes_.tolist(),'input_features':int(Xtr.shape[1]),
                         'model_family':'Torch-MLP','training_device':'cuda:0',
                         'required_preprocessing':'gpu_sentiment_tfidf.joblib'},
                         'preprocessor/inference_contract.json')
        print('GPU candidate MLflow run:',gpu_run.info.run_id)
else:
    print('GPU candidate skipped (CUDA unavailable or no trainable student). CPU baseline remains available.')


## E. Xuất toàn bộ dữ liệu, diagnostics, manifest và kết thúc MLflow run

MLflow lưu **dataset normalized + train/valid/test topic/sentiment + full scored predictions + topic keywords + configs + evaluation + model artifacts**. Với datasets có thông tin tác giả YouTube, file export không mang thông tin tác giả hoặc `author_hash` để tránh lưu dữ liệu không cần thiết. Mỗi dòng scored chứa `topic_id`, `sentiment_label`, `sentiment_method`, thời gian đăng và nguồn.

In [ ]:
# Independent manual labeling pool (for later assessment): không tuyên bố model đã chính xác.
# Ưu tiên 3 trạng thái dự đoán khác nhau; lấy gần đây + ngẫu nhiên, tránh author info.
pool=[]
for cls in ['negative','neutral','positive']:
    subset=docs.loc[docs.sentiment_label.eq(cls)].sort_values('published_at',ascending=False)
    pool.append(subset.head(40)[['doc_id','source_type','language','title','sentiment_input','published_at','sentiment_label']])
annotation=pd.concat(pool,ignore_index=True).drop_duplicates('doc_id')
annotation=annotation.rename(columns={'sentiment_label':'model_suggestion'})
annotation['sentiment_label']=''  # điền tay giá trị negative, neutral hoặc positive
annotation.to_csv(WORKDIR/'human_labeling_template.csv',index=False)
mlflow.log_artifact(str(WORKDIR/'human_labeling_template.csv'),'evaluation')

# Dataset cho dashboard, không chứa PII của người bình luận.
docs['published_at']=pd.to_datetime(docs['published_at'],utc=True).dt.tz_localize(None)
docs['event_date']=pd.to_datetime(docs['event_date'])
cols=['doc_id','source_type','published_at','event_date','days_old','title_clean','text_clean',
      'publisher','url','video_id','language','scope','engagement','topic_id','topic_name',
      'weak_sentiment','weak_confidence','sentiment_label','sentiment_confidence','sentiment_method']
scored=docs[cols].rename(columns={'title_clean':'title','text_clean':'text'}).copy()
scored['mlflow_run_id']=PARENT_RUN_ID
assert scored.doc_id.is_unique
assert scored.sentiment_label.isin(['positive','neutral','negative']).all()
assert scored.topic_id.notna().all()
scored.to_parquet(WORKDIR/'text_scored.parquet',index=False)
topic_lookup.to_parquet(WORKDIR/'topic_keywords.parquet',index=False)
ref_meta_path=WORKDIR/'wikipedia_reference_titles.parquet';REF_META.to_parquet(ref_meta_path,index=False)
# Snapshot scoring chỉ lưu cục bộ để audit; KHÔNG log toàn bộ prediction vào MLflow.
mlflow.log_artifact(str(WORKDIR/'topic_keywords.parquet'),'outputs')
mlflow.log_artifact(str(ref_meta_path),'outputs')
mlflow.log_input(mlflow.data.from_pandas(scored[['doc_id','source_type','sentiment_label','topic_id']],
                                      name='text_scored_full'),context='inference')
mlflow.log_metrics({'final_scored_rows':len(scored),
                    'recent7d_docs':int(scored.days_old.between(0,6).sum()),
                    'recent30d_docs':int(scored.days_old.between(0,29).sum()),
                    'source_count':scored.source_type.nunique(),
                    'pct_domestic_vn':float(scored.scope.eq('domestic_vn').mean())})
mlflow.log_dict({'run_id':PARENT_RUN_ID,'topic_k':BEST_K,'sources':FILES,
                 'column_names':list(scored.columns),'sentiment_is_verified':False,
                 'model_tracking_uri':MLFLOW_URI,'reference_date':str(ref_date.date())},
                'outputs/run_manifest.json')
mlflow.end_run()
print('✓ MLflow parent:',PARENT_RUN_ID)
print('✓ Output saved:',WORKDIR/'text_scored.parquet')
print('Next: temporal evaluation in the cells below (same notebook).')

## Kiểm thử và cách đọc kết quả

- **Topic:** silhouette cosine, kích thước cụm, trôi phân phối train–test (JS distance), từ khóa nổi bật. Topic không được gán ý nghĩa thủ công nếu chưa đọc bài đại diện.
- **Sentiment:** mô hình huấn luyện từ nhãn yếu → `pseudo_label_agreement_*`; **không suy ra accuracy/F1 thực tế**. File `human_labeling_template.csv` tạo mẫu gán nhãn để có validation chuẩn ở lần chạy tiếp.
- **Dữ liệu:** snapshot scored + split manifests + SHA-256 source đã được log trên parent MLflow; models và thông số từng thử nghiệm log trong nested run.
- **Chưa tự động đẩy vào PostgreSQL.** Notebook 02 tạo các mart để vẽ Text BI và hỗ trợ publish có kiểm soát.

---
## F. Temporal evaluation, topic trends và anomaly experiments

Phần này **dùng trực tiếp `scored` từ phần training**. Không cần notebook thứ hai, không tải snapshot từ MLflow. `scored` chỉ dùng tại phiên notebook để đánh giá; **không publish Data Warehouse**.


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from scipy.stats import entropy

OUTDIR=WORKDIR/'evaluation'; OUTDIR.mkdir(parents=True,exist_ok=True)
UPSTREAM_RUN_ID=PARENT_RUN_ID
MIN_TREND_RECENT=8
ALERT_NEGATIVE_RATE=0.50
TREND_SHARE_LIFT_MIN=1.50
ANOMALY_CONTAMINATION=0.08
scored=scored.copy()
required={'doc_id','source_type','published_at','event_date','scope','topic_id','sentiment_label','topic_name','language','engagement','title','url'}
assert not (required-set(scored.columns)), f'Missing columns: {required-set(scored.columns)}'
assert scored.doc_id.is_unique
scored['event_date']=pd.to_datetime(scored.event_date,utc=True,errors='coerce').dt.tz_localize(None).dt.normalize()
scored['published_at']=pd.to_datetime(scored.published_at,utc=True,errors='coerce').dt.tz_localize(None)
scored=scored.dropna(subset=['event_date']).copy()
REFERENCE_DATE=scored.event_date.max()
scored['age_days']=(REFERENCE_DATE-scored.event_date).dt.days
scored['window']=np.select([scored.age_days.between(0,6),scored.age_days.between(7,13)],
                           ['recent7','previous7'],default='older')
print('Temporal evaluation:',len(scored),'reference:',REFERENCE_DATE,'upstream:',UPSTREAM_RUN_ID)


## B. Kiểm tra thống kê và độ ổn định của Topic × Sentiment theo thời gian

Các phép nhóm dưới đây tạo **bảng chẩn đoán thí nghiệm trong RAM**, không phải mart dữ liệu và không được đưa thẳng lên dashboard. Mọi thay đổi của bộ lọc/ngưỡng đều log trong MLflow.

- `recent7` so với `previous7` sử dụng **ngày đăng nội dung**, không dùng ngày crawl.
- Topic growth xét tỷ trọng bên trong từng nguồn, giảm bớt thiên lệch do nguồn có quá nhiều bản ghi.
- Topic ID/names phải được phiên bản hóa cùng model upstream.


In [ ]:
# Giữ title, source, time, model metadata. Scope inferred is approximate.
assert scored.source_type.nunique()>=2
scored['is_positive']=scored.sentiment_label.eq('positive').astype(int)
scored['is_negative']=scored.sentiment_label.eq('negative').astype(int)
scored['is_neutral']=scored.sentiment_label.eq('neutral').astype(int)

def grouped_metrics(df, keys):
    v=df.groupby(keys,dropna=False).agg(
       docs=('doc_id','size'),positive=('is_positive','sum'),neutral=('is_neutral','sum'),
       negative=('is_negative','sum'),engagement_sum=('engagement','sum')).reset_index()
    v['positive_rate']=v.positive/v.docs.clip(lower=1)
    v['negative_rate']=v.negative/v.docs.clip(lower=1)
    v['neutral_rate']=v.neutral/v.docs.clip(lower=1)
    v['sentiment_balance']=(v.positive-v.negative)/v.docs.clip(lower=1)
    return v

mart_sentiment_daily=grouped_metrics(scored,['event_date','source_type','scope','language'])
mart_topic_daily=grouped_metrics(scored,['event_date','source_type','scope','topic_id','topic_name'])
mart_sentiment_topic=grouped_metrics(scored.loc[scored.window.eq('recent7')],
                                    ['source_type','scope','topic_id','topic_name'])
mart_kpi_daily=grouped_metrics(scored,['event_date'])
mart_kpi_daily['sources']=scored.groupby('event_date').source_type.nunique().reindex(mart_kpi_daily.event_date).values

In [ ]:
# Topic trend là tăng SHARE, không chỉ tăng count. Mỗi dòng = (nguồn, scope, topic).
recent=scored[scored.window.eq('recent7')].copy();prior=scored[scored.window.eq('previous7')].copy()
total_source_window=scored[scored.window.isin(['recent7','previous7'])].groupby(
    ['source_type','scope','window']).size().unstack(fill_value=0).reset_index()
for w in ['recent7','previous7']:
    if w not in total_source_window:total_source_window[w]=0

topics=scored[scored.window.isin(['recent7','previous7'])].groupby(
    ['source_type','scope','topic_id','topic_name','window'],dropna=False
).agg(docs=('doc_id','size'),negative=('is_negative','sum'),positive=('is_positive','sum')).reset_index()

trend_counts=topics.pivot_table(index=['source_type','scope','topic_id','topic_name'],
               columns='window',values=['docs','negative','positive'],aggfunc='sum',fill_value=0)
trend_counts.columns=['_'.join(v) for v in trend_counts.columns]
mart_topic_trends=trend_counts.reset_index()
for measure in ['docs','negative','positive']:
    for w in ['recent7','previous7']:
        col=f'{measure}_{w}'
        if col not in mart_topic_trends:mart_topic_trends[col]=0

mart_topic_trends=mart_topic_trends.merge(
    total_source_window.rename(columns={'recent7':'source_docs_recent7','previous7':'source_docs_previous7'}),
    on=['source_type','scope'],how='left',validate='many_to_one')
mart_topic_trends['share_recent7']=mart_topic_trends.docs_recent7/mart_topic_trends.source_docs_recent7.replace(0,np.nan)
mart_topic_trends['share_previous7']=mart_topic_trends.docs_previous7/mart_topic_trends.source_docs_previous7.replace(0,np.nan)
mart_topic_trends['share_delta_pp']=100*(mart_topic_trends.share_recent7.fillna(0)-mart_topic_trends.share_previous7.fillna(0))
mart_topic_trends['share_lift']=(mart_topic_trends.share_recent7.fillna(0)+0.001)/(mart_topic_trends.share_previous7.fillna(0)+0.001)
mart_topic_trends['negative_rate_recent7']=mart_topic_trends.negative_recent7/mart_topic_trends.docs_recent7.replace(0,np.nan)
mart_topic_trends['positive_rate_recent7']=mart_topic_trends.positive_recent7/mart_topic_trends.docs_recent7.replace(0,np.nan)
mart_topic_trends['trend_score']=mart_topic_trends.share_delta_pp*np.sqrt(mart_topic_trends.docs_recent7.clip(lower=0))
mart_topic_trends['sample_sufficient']=mart_topic_trends.docs_recent7.ge(MIN_TREND_RECENT)
mart_topic_trends['growing']=mart_topic_trends.sample_sufficient & mart_topic_trends.share_lift.ge(TREND_SHARE_LIFT_MIN)

# macro-average giảm ưu thế của YouTube comments có nhiều bản ghi hơn news.
mart_topic_trends_macro=mart_topic_trends.groupby(['scope','topic_id','topic_name'],dropna=False).agg(
    mean_share_delta_pp=('share_delta_pp','mean'),mean_share_lift=('share_lift','mean'),
    sources_observed=('source_type','nunique'),docs_recent7=('docs_recent7','sum'),
    docs_previous7=('docs_previous7','sum'),mean_neg_rate=('negative_rate_recent7','mean')
).reset_index()
mart_topic_trends_macro['cross_source_flag']=(mart_topic_trends_macro.sources_observed>=2)&(mart_topic_trends_macro.docs_recent7>=MIN_TREND_RECENT)
print('Topics trending:',int(mart_topic_trends.growing.sum()),'across sources:',len(mart_topic_trends_macro))

## C. Thử nghiệm anomaly detection cho temporal features

Đây là **candidate model**: Isolation Forest fit offline trên đặc trưng topic/ngày/nguồn. Khi số quan sát không đủ thì không fit, không đăng ký model giả. Kết quả chỉ được dùng để đánh giá thí nghiệm; pipeline của dự án sẽ xây dựng đặc trưng/inference riêng khi triển khai.


In [ ]:
# Build dense daily features for anomaly model; missing day => 0 volume (no observed documents).
daily_topic=mart_topic_daily.copy()
full_days=pd.date_range(scored.event_date.min(),REFERENCE_DATE,freq='D')
# Fit only on near-term 60-day period to prevent decade-long sparse YouTube history dominating model.
start_date=max(scored.event_date.min(),REFERENCE_DATE-pd.Timedelta(days=59))
recent_topic_days=daily_topic[daily_topic.event_date.ge(start_date)].copy()
recent_topic_days['volume_log1p']=np.log1p(recent_topic_days.docs)
features=['volume_log1p','negative_rate','positive_rate','sentiment_balance']
# Require enough daily observations; each row is topic/day/source, not necessarily consecutive days.
if len(recent_topic_days)>=80 and recent_topic_days.topic_id.nunique()>1:
    anomaly_pipe=Pipeline([('scale',StandardScaler()),('iforest',IsolationForest(
       n_estimators=160,max_samples='auto',contamination=ANOMALY_CONTAMINATION,
       random_state=SEED,n_jobs=-1))])
    recent_topic_days['split']=np.where(recent_topic_days.event_date<=REFERENCE_DATE-pd.Timedelta(days=7),'history','recent7')
    fit=recent_topic_days[recent_topic_days.split=='history']
    if len(fit)<50:fit=recent_topic_days
    anomaly_pipe.fit(fit[features].fillna(0))
    recent_topic_days['anomaly_score']=-anomaly_pipe.decision_function(recent_topic_days[features].fillna(0))
    recent_topic_days['is_anomaly']=(anomaly_pipe.predict(recent_topic_days[features].fillna(0))==-1)
    ANOMALY_MODEL_READY=True
else:
    recent_topic_days['anomaly_score']=np.nan
    recent_topic_days['is_anomaly']=False
    ANOMALY_MODEL_READY=False
mart_topic_anomaly=recent_topic_days[['event_date','source_type','scope','topic_id','topic_name','docs',
                                     'negative_rate','positive_rate','anomaly_score','is_anomaly']].copy()
mart_alerts=mart_topic_trends.loc[(mart_topic_trends.growing)
    & mart_topic_trends.negative_rate_recent7.ge(ALERT_NEGATIVE_RATE)].copy()
mart_alerts['alert_type']='negative_topic_burst'
mart_alerts['alert_reason']='Topic share grew and pseudo-negative ratio crossed configured threshold'
mart_alerts['reference_date']=REFERENCE_DATE
print('Anomaly model:',ANOMALY_MODEL_READY,'Rule alerts:',len(mart_alerts))

## D. Chẩn đoán entity mentions và domestic/international

Alias matching là **baseline xác định thực thể**, chưa phải mô hình NER đã đánh giá. Chỉ dùng so sánh nội bộ, kết hợp với cỡ mẫu và giới hạn dữ liệu; không kết luận rằng nguồn tiếng Việt luôn nói về bóng đá Việt Nam.


In [ ]:
# Danh sách alias mẫu minh bạch: có thể mở rộng từ entity_aliases.csv (entity_name, aliases pipe-separated, scope)
ENTITY_ENTRIES=[
 ('Manchester United','manchester united|man utd|man united|quỷ đỏ','international'),
 ('Manchester City','manchester city|man city','international'),
 ('Liverpool','liverpool','international'),('Arsenal','arsenal','international'),
 ('Chelsea','chelsea','international'),('Real Madrid','real madrid','international'),
 ('Barcelona','barcelona|barca|barça','international'),('Bayern Munich','bayern munich|bayern','international'),
 ('PSG','psg|paris saint-germain','international'),('Tottenham','tottenham|spurs','international'),
 ('Đội tuyển Việt Nam','đội tuyển việt nam|tuyển việt nam|vietnam national team|u23 việt nam','domestic_vn'),
 ('V-League','v-league|vleague|v.league','domestic_vn'),
 ('Hà Nội FC','hà nội fc|hanoi fc','domestic_vn'),
 ('Hoàng Anh Gia Lai','hoàng anh gia lai|hagl','domestic_vn'),
 ('Nam Định','thép xanh nam định|nam định fc','domestic_vn'),
 ('Cristiano Ronaldo','cristiano ronaldo|ronaldo','international'),
 ('Lionel Messi','lionel messi|messi','international'),
 ('Champions League','champions league|uefa champions league|cúp c1','international')]
ALIAS_PATH=WORKDIR/'entity_aliases.csv'
if ALIAS_PATH.exists():
    aliases_df=pd.read_csv(ALIAS_PATH).fillna('')
    assert {'entity_name','aliases','scope'}.issubset(aliases_df.columns)
else:
    aliases_df=pd.DataFrame(ENTITY_ENTRIES,columns=['entity_name','aliases','scope'])
    aliases_df.to_csv(ALIAS_PATH,index=False)

compiled=[]
for row in aliases_df.itertuples(index=False):
    for alias in str(row.aliases).split('|'):
        a=alias.strip()
        if len(a)>2:
            compiled.append((row.entity_name,row.scope,re.compile(r'(?<!\w)'+re.escape(a)+r'(?!\w)',flags=re.IGNORECASE)))

entity_records=[]
for d in scored[['doc_id','event_date','source_type','title','text','topic_id','sentiment_label','scope']].itertuples(index=False):
    snippet=(str(d.title)+' '+str(d.text)[:600]).lower()
    matched={}
    for name,scope,rx in compiled:
        if rx.search(snippet):matched[name]=scope
    for name,scope in matched.items():
        entity_records.append({'doc_id':d.doc_id,'event_date':d.event_date,'source_type':d.source_type,
                    'entity_name':name,'entity_scope':scope,'topic_id':d.topic_id,'sentiment_label':d.sentiment_label})
mentions=pd.DataFrame(entity_records,columns=['doc_id','event_date','source_type','entity_name',
                   'entity_scope','topic_id','sentiment_label'])
if len(mentions):
    mentions['is_positive']=mentions.sentiment_label.eq('positive').astype(int)
    mentions['is_negative']=mentions.sentiment_label.eq('negative').astype(int)
    mentions['is_neutral']=mentions.sentiment_label.eq('neutral').astype(int)
    mart_entity_daily=grouped_metrics(mentions.assign(engagement=0),
                           ['event_date','source_type','entity_name','entity_scope'])
else:
    mart_entity_daily=pd.DataFrame(columns=['event_date','source_type','entity_name','entity_scope',
                             'docs','positive','neutral','negative','engagement_sum','positive_rate',
                             'negative_rate','neutral_rate','sentiment_balance'])
scored['display_scope']=scored['scope']
# chỉ gán scope từ entity nếu scope gốc unknown và entity rõ ràng, ưu tiên xác định Việt Nam nếu có.
if len(mentions):
    dom=set(mentions.loc[mentions.entity_scope.eq('domestic_vn'),'doc_id'])
    intl=set(mentions.loc[mentions.entity_scope.eq('international'),'doc_id'])
    mask=scored.scope.eq('unknown')
    scored.loc[mask & scored.doc_id.isin(dom),'display_scope']='domestic_vn'
    scored.loc[mask & scored.doc_id.isin(intl) & ~scored.doc_id.isin(dom),'display_scope']='international'
print('Entity mentions:',len(mentions),'unique docs:',mentions.doc_id.nunique() if len(mentions) else 0)

## E. Đánh giá offline và log MLflow (không tạo mart)

Ghi các **evaluation artifacts**: tỷ lệ sentiment theo nguồn, topic stability, trend candidates, anomaly candidates, entity coverage, source freshness và evaluation manifest. Đây là diagnostic snapshots chỉ để phân tích trên MLflow, **không có contract bảng DWH**. Để model anomaly dùng được trong project, cần promotion qua Model Registry sau khi kiểm định thủ công; notebook **không tự gán `champion`**.


In [ ]:
# Chỉ tạo diagnostic outputs — không xuất mart BI/SQL và không publish database.
source_health = scored.groupby('source_type').agg(
    total_docs=('doc_id','size'), min_published_at=('published_at','min'),
    max_published_at=('published_at','max'), distinct_days=('event_date','nunique'),
    language_variants=('language','nunique')).reset_index()
recent_counts = scored.loc[scored.window.eq('recent7')].source_type.value_counts()
source_health['recent7_docs'] = source_health.source_type.map(recent_counts).fillna(0).astype(int)
source_health['recency_days'] = (REFERENCE_DATE-source_health.max_published_at.dt.normalize()).dt.days

# Không log raw user comments/text vào evaluation run; các file chỉ có statistics/metadata.
REPORTS = {
    'source_health': source_health,
    'sentiment_daily_diagnostic': mart_sentiment_daily,
    'topic_daily_diagnostic': mart_topic_daily,
    'topic_trend_diagnostic': mart_topic_trends,
    'topic_cross_source_diagnostic': mart_topic_trends_macro,
    'topic_sentiment_diagnostic': mart_sentiment_topic,
    'anomaly_diagnostic': mart_topic_anomaly,
    'negative_burst_rule_diagnostic': mart_alerts,
    'entity_coverage_diagnostic': mart_entity_daily,
}
for name,df in REPORTS.items():
    df.to_parquet(OUTDIR/f'{name}.parquet',index=False)
    print(name,df.shape)

def safe_rate(n,d): return float(n/d) if d else float('nan')
metrics = {
    'eval_scored_docs': len(scored),
    'eval_recent7_docs': len(recent),
    'eval_previous7_docs': len(prior),
    'eval_topic_count': int(scored.topic_id.nunique()),
    'eval_topic_growing_count': int(mart_topic_trends.growing.sum()),
    'eval_topic_sample_sufficient_count': int(mart_topic_trends.sample_sufficient.sum()),
    'eval_negative_burst_alerts': len(mart_alerts),
    'eval_anomaly_count': int(mart_topic_anomaly.is_anomaly.sum()),
    'eval_anomaly_rows': len(mart_topic_anomaly),
    'eval_entity_mentions': len(mentions),
    'eval_entity_doc_coverage': safe_rate(mentions.doc_id.nunique(),len(scored)),
    'eval_unknown_scope_share': float(scored.display_scope.eq('unknown').mean()),
    'eval_negative_label_share': float(scored.sentiment_label.eq('negative').mean()),
    'eval_neutral_label_share': float(scored.sentiment_label.eq('neutral').mean()),
}
# Lưu manifest rõ provenance và coverage của các report.
manifest = {
    'role':'offline_experiment_evaluation_only',
    'upstream_run_id':UPSTREAM_RUN_ID,
    'reference_date':str(REFERENCE_DATE.date()),
    'training_dataset_type':'topic_day_source_aggregates',
    'anomaly_features':features,
    'anomaly_training_ready':bool(ANOMALY_MODEL_READY),
    'sentiment_labels':'weak_or_pseudo_labels_not_ground_truth',
    'reports':{name:{'rows':len(df),'columns':df.columns.tolist()} for name,df in REPORTS.items()},
    'production_outputs':'NONE — produced by football-lake project',
}
(OUTDIR/'evaluation_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
print('Evaluation metrics:',metrics)


In [ ]:
# Mỗi run là một phiên bản đánh giá tái lập được; không ghi DWH.
with mlflow.start_run(run_name=f'Football_Text_Temporal_Evaluation_{REFERENCE_DATE:%Y%m%d}') as run:
    EVAL_RUN_ID=run.info.run_id
    mlflow.set_tags({
        'project':'football-lake',
        'pipeline':'offline-model-evaluation',
        'stage':'research',
        'upstream_run_id':UPSTREAM_RUN_ID or 'local_snapshot',
        'sentiment_status':'weak/pseudo labels; validation required',
        'production_publish':'false',
        'linked_training_run_id':PARENT_RUN_ID,
        'reference_date':str(REFERENCE_DATE.date()),
    })
    mlflow.log_params({
        'window_recent_days':7,
        'window_previous_days':7,
        'min_recent_for_trend':MIN_TREND_RECENT,
        'min_topic_share_lift':TREND_SHARE_LIFT_MIN,
        'negative_alert_rate_threshold':ALERT_NEGATIVE_RATE,
        'anomaly_algorithm':'IsolationForest',
        'anomaly_contamination':ANOMALY_CONTAMINATION,
        'anomaly_random_state':SEED,
        'anomaly_training_device':'cpu (scikit-learn)',
        'gpu_detected':str(GPU_AVAILABLE),
        'anomaly_model_ready':str(bool(ANOMALY_MODEL_READY)).lower(),
        'anomaly_feature_list':','.join(features),
    })
    # Ghi hash/schema/count của scored input; không bắt buộc lưu toàn bộ nội dung text.
    mlflow.log_input(mlflow.data.from_pandas(
        scored[['doc_id','source_type','sentiment_label','topic_id']],
        name='text_scored_evaluation_snapshot',source=UPSTREAM_RUN_ID or 'local_snapshot'),
        context='evaluation')
    mlflow.log_metrics({k:v for k,v in metrics.items() if np.isfinite(v)})
    mlflow.log_artifacts(str(OUTDIR),artifact_path='evaluation')
    if ALIAS_PATH.exists():
        mlflow.log_artifact(str(ALIAS_PATH),artifact_path='evaluation/entity_aliases')
    if ANOMALY_MODEL_READY:
        mlflow.sklearn.log_model(anomaly_pipe,name='anomaly_candidate',
            input_example=fit[features].head(min(3,len(fit))))
        mlflow.log_input(mlflow.data.from_pandas(
            fit[['event_date','topic_id','source_type','docs']],name='anomaly_fit_snapshot'),
            context='training')
    print('MLflow evaluation run:',EVAL_RUN_ID)
print('Done. No SQL, marts, PostgreSQL writes or production inference from this notebook.')


## F. Bàn giao cho dự án `football-lake`

1. Dự án tải **các phiên bản model đã được duyệt** từ MLflow Registry (`models:/<name>@champion` chỉ khi alias thật sự tồn tại). Notebook 01 hiện cần bổ sung bước đăng ký và phê duyệt model nếu chưa có.
2. Pipeline dự án đọc các bảng text mới từ DWH/lake, inference chủ đề/sentiment, tự xây dựng temporal features và gọi anomaly model phù hợp với schema/version của nó.
3. Pipeline dự án ghi `model_name`, `model_version`, `mlflow_run_id`, `inference_at`, `data_snapshot_id`, `published_at` và trạng thái chất lượng vào output để truy vết.
4. Pipeline dự án tự tính các bảng phục vụ Superset (`text_documents`, `text_topic_daily`, `text_sentiment_daily`, `text_topic_trends`, `text_negative_alerts`, ...) và publish vào DWH; **không sử dụng notebook này như một ETL/BI mart job**.
5. Chưa đánh giá bằng ground-truth thì không tự promote sentiment/anomaly lên `champion` và không khẳng định sentiment là độ chính xác thật.

Nếu muốn kiểm tra kết quả một run, xem tab Artifacts/Models/Metrics/Parameters của experiment `Football_Text_Intelligence_360` trên MLflow.


## Kết thúc — Cách sử dụng trong dự án

- **MLflow:** sử dụng Parent Run ID (`PARENT_RUN_ID`) và Evaluation Run ID (`EVAL_RUN_ID`) để theo dõi thí nghiệm, các model artifacts, parameters, metrics và báo cáo. `topic_keywords.parquet` + alias dictionary là metadata cần version cùng model khi đưa vào pipeline.
- **Model Registry:** log model vào run **không đồng nghĩa** đã đăng ký production; phải kiểm tra chất lượng, đăng ký model version và gán alias `champion` trước khi code dự án dùng `models:/...@champion`.
- **Inference production:** dự án đọc text từ MinIO, tải đúng model/preprocessor từ MLflow, tính predictions mới và xuất Parquet/DWH. Không dùng `text_scored.parquet` của notebook như bảng production.
- **GPU:** phần neural teacher tận dụng CUDA khi bật; các sklearn training CPU-only và MLflow ghi lại môi trường chạy. Không tự động tuyên bố đã train deep learning model nếu chỉ dùng teacher suy luận.
